In [15]:
import pandas as pd
import numpy as np
from pathlib import Path

# =============================================================================
# INPUT FILES
# =============================================================================

HAZARD_CSV = Path("data") / "hazard.csv"
EXPOSURE_CSV = Path("data") / "exposure.csv"
VULNERABILITY_CSV = Path("data") / "vulnerability.csv"
GOVT_CSV = Path("data") / "government_response.csv"

hazard = pd.read_csv(HAZARD_CSV)
exposure = pd.read_csv(EXPOSURE_CSV)
vulnerability = pd.read_csv(VULNERABILITY_CSV)
gov = pd.read_csv(GOVT_CSV)

# Keep only required columns
hazard = hazard[["district", "timeperiod", "heat_hazard"]]
exposure = exposure[["district", "timeperiod", "exposure"]]
vulnerability = vulnerability[["district", "timeperiod", "vulnerability_class"]]
gov = gov[["district", "timeperiod", "government_response"]]

# =============================================================================
# MERGE COMPONENTS (DISTRICT × TIMEPERIOD)
# =============================================================================

df = hazard.merge(exposure, on=["district", "timeperiod"], how="inner")
df = df.merge(vulnerability, on=["district", "timeperiod"], how="inner")
df = df.merge(gov, on=["district", "timeperiod"], how="inner")

# =============================================================================
# WEIGHTS
# =============================================================================

weights = {
    "heat_hazard": 4,
    "exposure": 1,
    "vulnerability_class": 2,
    "government_response": 2
}

# =============================================================================
# TOPSIS PER TIMEPERIOD (MONTHWISE COMPARISON)
# =============================================================================

results = []

for tp, g in df.groupby("timeperiod"):
    g = g.copy()

    # -------------------------
    # MIN-MAX NORMALIZATION
    # -------------------------
    norm = pd.DataFrame()

    for col in weights:
        min_v = g[col].min()
        max_v = g[col].max()

        if max_v == min_v:
            norm[col] = 0
        else:
            norm[col] = (g[col] - min_v) / (max_v - min_v)

    # -------------------------
    # APPLY WEIGHTS
    # -------------------------
    for col in weights:
        norm[col] = norm[col] * (weights[col] / sum(weights.values()))

    # -------------------------
    # IDEAL BEST / WORST
    # -------------------------
    ideal_best = norm.max()
    ideal_worst = norm.min()

    # -------------------------
    # DISTANCES
    # -------------------------
    dist_best = np.sqrt(((norm - ideal_best) ** 2).sum(axis=1))
    dist_worst = np.sqrt(((norm - ideal_worst) ** 2).sum(axis=1))

    # -------------------------
    # TOPSIS SCORE
    # -------------------------
    g["topsis_score"] = dist_worst / (dist_best + dist_worst)

    results.append(g)

# Combine all months
df = pd.concat(results, ignore_index=True)

# =============================================================================
# RISK CLASSIFICATION (MONTHWISE FIXED THRESHOLDS)
# =============================================================================

def classify(x):
    if x <= 0.2:
        return 1
    elif x <= 0.4:
        return 2
    elif x <= 0.6:
        return 3
    elif x <= 0.8:
        return 4
    else:
        return 5

df["risk_class"] = df["topsis_score"].apply(classify)

# =============================================================================
# SAVE OUTPUT
# =============================================================================

OUTPUT = Path("data") / "final_risk_score.csv"
df.to_csv(OUTPUT, index=False)

print("Saved:", OUTPUT)

# =============================================================================
# SUMMARY
# =============================================================================

print("\nTOPSIS Summary:")
print(df["topsis_score"].describe())

print("\nRisk Class Distribution:")
print(df["risk_class"].value_counts().sort_index())

print("\nPreview:")
print(df[["district", "timeperiod", "topsis_score", "risk_class"]].head())



# =============================================================================
# APPEND risk_class TO MASTER_VARIABLES.csv
# =============================================================================

master_path = Path("data") / "MASTER_VARIABLES.csv"

master = pd.read_csv(master_path)

master["district"] = master["district"].astype(str).str.strip()
master["timeperiod"] = master["timeperiod"].astype(str).str.strip()

df["district"] = df["district"].astype(str).str.strip()
df["timeperiod"] = df["timeperiod"].astype(str).str.strip()

if "risk_class" in master.columns:
    master = master.drop(columns=["risk_class"])

master = master.merge(
    df[["district", "timeperiod", "risk_class"]],
    on=["district", "timeperiod"],
    how="left"
)

master.to_csv(master_path, index=False)

print("risk_class appended to MASTER_VARIABLES.csv")

Saved: data/final_risk_score.csv

TOPSIS Summary:
count    7222.000000
mean        0.408014
std         0.192312
min         0.000000
25%         0.303832
50%         0.416526
75%         0.561474
max         0.936838
Name: topsis_score, dtype: float64

Risk Class Distribution:
risk_class
1    1112
2    2127
3    2968
4     929
5      86
Name: count, dtype: int64

Preview:
  district timeperiod  topsis_score  risk_class
0   Anugul    2023_01      0.490256           3
1   Anugul    2023_01      0.490256           3
2   Anugul    2023_01      0.490256           3
3   Anugul    2023_01      0.490256           3
4   Anugul    2023_01      0.490256           3
risk_class appended to MASTER_VARIABLES.csv


In [11]:
from pathlib import Path

print("CWD:", Path.cwd())
print("Hazard exists:", Path("data/hazard.csv").exists())
print("Exposure exists:", Path("data/exposure.csv").exists())
print("Vulnerability exists:", Path("data/vulnerability.csv").exists())

CWD: /home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/RiskScoreModel
Hazard exists: True
Exposure exists: True
Vulnerability exists: True
